In [18]:
import requests
import json
from datetime import datetime, timezone
import random
import asyncio, aiohttp
import aiolimiter
from tqdm.auto import tqdm


In [19]:
def load_posts_dict(file_path):
    """
    Reads a .jsonl file where each line is a JSON object representing a post.
    Returns a dict mapping post_id (or URI) -> post data.

    Supports multiple Bluesky data formats:
      - post["uri"]
      - post["post"]["uri"]
      - post["id"]
    """
    posts_dict = {}
    bad_lines = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                post = json.loads(line)
            except json.JSONDecodeError:
                bad_lines += 1
                continue

            # Try to find a reliable ID/URI key
            post_id = (
                post.get("uri")
                or post.get("post", {}).get("uri")
                or post.get("id")
            )

            posts_dict[post_id] = post

    print(f"Loaded {len(posts_dict)} posts from {file_path} ({bad_lines} bad lines skipped)")
    return posts_dict


In [ ]:
# =====================================================
# BLUESKY DATASET BUILDER (posts_dict -> users)
# FINAL VERSION (timezone-safe, progress bars, stable)
# =====================================================


REPOST_API  = "https://public.api.bsky.app/xrpc/app.bsky.feed.getRepostedBy"
PROFILE_API = "https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile"
FEED_API    = "https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed"
FOLLOW_API  = "https://public.api.bsky.app/xrpc/app.bsky.graph.getFollows"

HEADERS = {"User-Agent": "repost-prediction-research/1.0"}

# -------------------------
# UTILS (TIMEZONE SAFE)
# -------------------------

def parse_dt(ts: str):
    """Parse Bluesky timestamp into UTC-aware datetime."""
    return datetime.fromisoformat(ts.replace("Z", "+00:00")).astimezone(timezone.utc)

def get_author_dids(posts_dict):
    authors = set()
    for post in posts_dict.values():
        did = (
            post.get("author", {}).get("did")
            or post.get("post", {}).get("author", {}).get("did")
        )
        if did:
            authors.add(did)
    return authors

# -------------------------
# FETCH REPOSTERS
# -------------------------

async def fetch_reposters(session, uri):
    params = {"uri": uri, "limit": 100}
    try:
        async with session.get(REPOST_API, params=params, headers=HEADERS) as r:
            if r.status != 200:
                return []
            data = await r.json()
            return [u["did"] for u in data.get("repostedBy", [])]
    except Exception:
        return []

async def collect_reposter_dict(posts_dict, concurrency=25):
    posts, tasks = [], []

    connector = aiohttp.TCPConnector(limit_per_host=concurrency)
    async with aiohttp.ClientSession(connector=connector) as session:
        for uri, post in posts_dict.items():
            if post.get("repostCount", 0) > 0:
                posts.append(uri)
                tasks.append(fetch_reposters(session, uri))

        reposter_dict = {}

        for uri, task in zip(
            tqdm(posts, desc="Fetching reposters", unit="post"),
            asyncio.as_completed(tasks)
        ):
            reposters = await task
            for r in reposters:
                reposter_dict.setdefault(r, []).append(uri)

    return reposter_dict

# -------------------------
# YOUR FOLLOW FETCHING (UNCHANGED LOGIC)
# -------------------------

async def fetch_follows(session, reposter, author_set, retries=3, first_page_only=True):
    follows = set()
    cursor = None
    headers = {"User-Agent": "Mozilla/5.0"}

    delay = 1
    for attempt in range(retries):
        try:
            while True:
                params = {"actor": reposter, "limit": 100}
                if cursor:
                    params["cursor"] = cursor

                async with session.get(FOLLOW_API, params=params, headers=headers) as r:
                    if r.status != 200:
                        if 500 <= r.status < 600 and attempt < retries - 1:
                            await asyncio.sleep(delay)
                            delay *= 2
                            continue
                        return reposter, []

                    data = await r.json()
                    follows.update(
                        u["did"] for u in data.get("follows", [])
                        if u.get("did") in author_set
                    )

                    cursor = data.get("cursor")
                    if not cursor or first_page_only:
                        break
            return reposter, list(follows)

        except Exception:
            if attempt < retries - 1:
                await asyncio.sleep(delay)
                delay *= 2
                continue
            return reposter, []

async def collect_reposter_followed_authors(
    reposter_dict,
    posts_dict,
    concurrency=100,
    reqs_per_sec=50
):
    reposters = list(reposter_dict.keys())
    total = len(reposters)
    author_set = get_author_dids(posts_dict)
    counter = {"done": 0}

    rate_limiter = aiolimiter.AsyncLimiter(reqs_per_sec, 1)
    connector = aiohttp.TCPConnector(limit=300, limit_per_host=50, ttl_dns_cache=300)
    timeout = aiohttp.ClientTimeout(total=10, connect=5, sock_read=5)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        sem = asyncio.Semaphore(concurrency)

        async def limited_fetch(reposter):
            async with sem, rate_limiter:
                rep, authors = await fetch_follows(session, reposter, author_set)
                counter["done"] += 1
                if counter["done"] % 100 == 0 or counter["done"] == total:
                    print(
                        f"\rFetching followed authors: "
                        f"{counter['done']}/{total} "
                        f"({counter['done']/total:.1%})",
                        end="",
                        flush=True
                    )
                return rep, authors

        tasks = [limited_fetch(r) for r in reposters]
        responses = await asyncio.gather(*tasks)

    print()
    return {r: authors for r, authors in responses if authors}

# -------------------------
# FETCH PROFILE
# -------------------------

async def fetch_profile(session, did):
    async with session.get(PROFILE_API, params={"actor": did}, headers=HEADERS) as r:
        if r.status != 200:
            return None
        return await r.json()

# -------------------------
# FETCH USER HISTORY (≤50 POSTS / REPOSTS)
# -------------------------

async def fetch_user_history(session, did, limit=50):
    """
    Fetch up to `limit` historical POSTS or REPOSTS only.
    Replies and quotes are ignored.
    """
    history = []
    cursor = None

    while len(history) < limit:
        params = {"actor": did, "limit": min(100, limit - len(history))}
        if cursor:
            params["cursor"] = cursor

        async with session.get(FEED_API, params=params, headers=HEADERS) as r:
            if r.status != 200:
                break

            data = await r.json()
            feed = data.get("feed", [])

            for item in feed:
                post = item.get("post")
                if not post:
                    continue

                record = post.get("record", {})
                reason = item.get("reason", {})

                # -------------------------
                # REPOST
                # -------------------------
                if reason.get("$type", "").endswith("reasonRepost"):
                    activity_type = "repost"
                    parent_post_uri = post["uri"]
                    parent_author_did = post["author"]["did"]
                    reposted_at = reason.get("indexedAt")

                # -------------------------
                # ORIGINAL POST (not reply, not quote)
                # -------------------------
                elif "reply" not in record:
                    activity_type = "post"
                    parent_post_uri = None
                    parent_author_did = None
                    reposted_at = None

                else:
                    # Ignore replies & quotes
                    continue

                # -------------------------
                # CONTENT FEATURES
                # -------------------------
                facets = record.get("facets", [])
                has_links = any(
                    f["features"][0]["$type"].endswith("link")
                    for f in facets
                )

                embed = record.get("embed", {})
                media_type = None
                media_count = 0
                if embed:
                    et = embed.get("$type", "")
                    if et.endswith("images"):
                        media_type = "image"
                        media_count = len(embed.get("images", []))
                    elif et.endswith("video"):
                        media_type = "video"
                        media_count = 1
                    elif et.endswith("external"):
                        media_type = "external"
                        media_count = 1
                    elif et.endswith("recordWithMedia"):
                        media_type = "mixed"
                        media_count = 1

                # -------------------------
                # BUILD RECORD
                # -------------------------
                history.append({
                    "activity_type": activity_type,
                    "created_at": record.get("createdAt"),
                    "reposted_at": reposted_at,

                    "post_uri": post["uri"],
                    "post_author_did": post["author"]["did"],

                    "parent_post_uri": parent_post_uri,
                    "parent_author_did": parent_author_did,

                    "text": record.get("text", ""),
                    "langs": record.get("langs", []),

                    "like_count": post.get("likeCount"),
                    "repost_count": post.get("repostCount"),
                    "reply_count": post.get("replyCount"),
                    "quote_count": post.get("quoteCount"),

                    "has_links": has_links,
                    "media_type": media_type,
                    "media_count": media_count,

                    "labels": post.get("labels", []),
                })

                if len(history) >= limit:
                    break

            cursor = data.get("cursor")
            if not cursor:
                break

    return history


# -------------------------
# MAIN BUILDER
# -------------------------

async def build_users_from_posts(posts_dict):
    reposter_dict = await collect_reposter_dict(posts_dict)
    followed_authors = await collect_reposter_followed_authors(
        reposter_dict, posts_dict
    )

    user_dids = set(reposter_dict.keys())
    user_dids |= get_author_dids(posts_dict)

    users = {}

    async with aiohttp.ClientSession() as session:

        async def process_user(did):
            profile = await fetch_profile(session, did)
            if not profile:
                return

            history = await fetch_user_history(session, did)

            created = profile.get("createdAt")
            age_days = (
                max(1, (datetime.now(timezone.utc) - parse_dt(created)).days)
                if created else None
            )

            users[did] = {
                "profile": {
                    "did": did,
                    "handle": profile.get("handle"),
                    "display_name": profile.get("displayName"),
                    "description": profile.get("description"),
                    "created_at": created,
                },
                "stats": {
                    "followers": profile.get("followersCount"),
                    "follows": profile.get("followsCount"),
                    "posts": profile.get("postsCount"),
                    "account_age_days": age_days,
                },
                "history": history,
                "reposted_posts": reposter_dict.get(did, []),
                "follows_authors": followed_authors.get(did, []),
            }

        tasks = [process_user(d) for d in user_dids]

        for task in tqdm(
            asyncio.as_completed(tasks),
            total=len(tasks),
            desc="Building users",
            unit="user"
        ):
            await task

    return users

In [20]:
posts_dict = load_posts_dict("new.jsonl")   

Loaded 5000 posts from new.jsonl (0 bad lines skipped)


In [ ]:
users = await build_users_from_posts(posts_dict)

Fetching reposters: 100%|██████████| 1028/1028 [00:12<00:00, 80.16post/s]


Fetching followed authors: 2658/2658 (100.0%)


Building users: 100%|██████████| 4529/4529 [04:56<00:00, 15.28user/s]


In [25]:
with open("users.json", "w", encoding="utf-8") as f:
    json.dump(users, f, ensure_ascii=False, indent=2)
